In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA silver;
SET TIME ZONE 'UTC';

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.silver.tmp_characters_history_enriched AS
WITH filtered_state AS (
  SELECT character_id,
         name,
         valid_from,
         valid_to
    FROM tibia_analytics.silver.characters_state
   WHERE name_type = 'observed_current'
     AND is_active_lock = TRUE
),
filtered_identity AS (
  SELECT character_id,
         deleted_at
    FROM tibia_analytics.silver.characters_identity
),
filtered_overrides AS (
  SELECT snapshot_id,
         name,
         source_file_date,
         character_id
    FROM tibia_analytics.silver.characters_identity_overrides
)
SELECT /*+ BROADCAST(fs, fi, fo) */
       ch.snapshot_id,
       COALESCE(fo.character_id, fs.character_id) AS character_id,
       ch.name,
       ch.traded,
       fi.deleted_at AS deletion_at,
       ch.former_names,
       ch.title,
       ch.unlocked_titles,
       ch.sex,
       ch.vocation,
       CASE
         WHEN ch.vocation LIKE '%Druid%'    THEN 'Druid'
         WHEN ch.vocation LIKE '%Knight%'   THEN 'Knight'
         WHEN ch.vocation LIKE '%Monk%'     THEN 'Monk'
         WHEN ch.vocation LIKE '%Paladin%'  THEN 'Paladin'
         WHEN ch.vocation LIKE '%Sorcerer%' THEN 'Sorcerer'
         ELSE ch.vocation
       END AS vocation_normalized,
       ch.level,
       ch.achievement_points,
       ch.world_id,
       ch.world,
       ch.former_worlds,
       ch.residence,
       ch.married_to,
       ch.houses,
       ch.guild,
       ch.last_login_at,
       ch.comment,
       ch.position,
       ch.account_status,
       ch.account_badges,
       ch.achievements,
       ch.deaths_truncated,
       ch.deaths,
       ch.account_information,
       ch.other_characters,
       ch.source_file_date,
       ch.api_timestamp,
       current_timestamp() AS created_at,
       current_timestamp() AS processed_at
  FROM tibia_analytics.silver.characters_history AS ch
  LEFT JOIN filtered_overrides AS fo
    ON fo.snapshot_id      = ch.snapshot_id
   AND fo.name             = ch.name
   AND fo.source_file_date = ch.source_file_date
  LEFT JOIN filtered_state AS fs
    ON fs.name             = ch.name
   AND ch.source_file_date BETWEEN fs.valid_from AND fs.valid_to
  LEFT JOIN filtered_identity AS fi
    ON fi.character_id     = COALESCE(fo.character_id, fs.character_id);

In [0]:
CREATE OR REPLACE TABLE tibia_analytics.silver.tmp_characters_history_enriched_qualified AS
 SELECT *
   FROM tibia_analytics.silver.tmp_characters_history_enriched
QUALIFY ROW_NUMBER() OVER (
          PARTITION BY character_id, source_file_date
              ORDER BY api_timestamp DESC
        ) = 1;

In [0]:
MERGE INTO tibia_analytics.silver.characters_history_enriched           AS target
USING tibia_analytics.silver.tmp_characters_history_enriched_qualified  AS source
   ON target.snapshot_id      = source.snapshot_id
  AND target.character_id     = source.character_id
  AND target.source_file_date = source.source_file_date
 WHEN NOT MATCHED THEN INSERT *;

In [0]:
DROP TABLE IF EXISTS tibia_analytics.silver.tmp_characters_history_enriched;
DROP TABLE IF EXISTS tibia_analytics.silver.tmp_characters_history_enriched_qualified;